# Minimum LFP Viewer

Load a Maxwell recording, run a SpikeInterface preprocessing chain
(bandpass + notch line-noise removal + common median reference), cache the
LFP signal to disk, then explore it with an interactive electrode selector
and the SpikeInterface `plot_traces` widget.

**Steps**
1. Configure raw data path, output cache directory, well, and notch frequencies.
2. Build the preprocessing chain and save the LFP as Zarr (skips on cache hit).
3. Pick electrodes by clicking / box-selecting on the chip layout.
4. Plot the selected traces.

## 1. Configuration

In [ ]:
# --- USER INPUTS -------------------------------------------------------------
data_path = "/mnt/benshalom-nas/raw_data/rbs_maxtwo_desktop/harddisk24tbvol1/KCNT1_MEASlices_06062025_CA_PS/250606/M07896/Network/000003/data.raw.h5"
output_dir = "./lfp_cache"   # where the Zarr cache + metadata get written
well_id = "well004"           # Maxwell stream id (well000 .. well023)
rec_name = None               # leave None unless the file has multiple recs per well

# Notch frequencies
notch_electricity_hz = 60     # 50 (EU) or 60 (US); harmonics are added automatically
notch_light_hz = None         # set e.g. 470 or any opto carrier; leave None to skip
notch_q = 30.0                # notch quality factor (higher = narrower)

# LFP band
lfp_low_hz = 0.5
lfp_high_hz = 300.0

# Downsample raw before filtering. Maxwell records at 20 kHz, but LFP analysis
# only needs ~2 * lfp_high_hz of bandwidth, so we decimate to 1 kHz by default
# for a ~20x speedup on every filter/spectrum pass. Set to None to keep the
# native sample rate.
lfp_target_fs_hz = 1000.0

# Overwrite cache if it already exists?
overwrite_cache = False
# -----------------------------------------------------------------------------

import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_dir = Path(output_dir).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Raw data : {data_path}")
print(f"Cache dir: {output_dir}")
print(f"Well     : {well_id}")
print(f"Notch    : electricity={notch_electricity_hz} Hz, light={notch_light_hz} Hz")
print(f"Target fs: {lfp_target_fs_hz} Hz" if lfp_target_fs_hz else "Target fs: native (no resample)")


## 2. Preprocess and cache LFP

Builds the lazy chain: `unsigned_to_signed → bandpass → notch(line + harmonics) → [notch(light)] → common median reference`, then saves a Zarr cache. Reuses the cache on subsequent runs unless `overwrite_cache=True`.

In [ ]:
import json
import shutil
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

from acute_slice_mea.recording import load_maxwell_recording


def build_lfp_chain(raw, *, target_fs_hz, electricity_hz, light_hz, q, low_hz, high_hz):
    """Optional resample + bandpass + line-noise notch (+ harmonics) + optional opto notch + CMR."""
    rec = spre.unsigned_to_signed(raw)
    native_fs = float(rec.get_sampling_frequency())
    if target_fs_hz is not None and float(target_fs_hz) < native_fs:
        rec = spre.resample(rec, resample_rate=int(target_fs_hz), dtype="float32")
    rec = spre.bandpass_filter(
        rec,
        freq_min=low_hz,
        freq_max=high_hz,
        margin_ms=10000,
        ignore_low_freq_error=True,
        dtype="float32",
    )
    nyq = rec.get_sampling_frequency() / 2
    notch_freqs = [electricity_hz, 2 * electricity_hz, 3 * electricity_hz]
    if light_hz:
        notch_freqs.append(float(light_hz))
    for f in notch_freqs:
        if 0 < f < nyq:
            rec = spre.notch_filter(rec, freq=f, q=q)
    rec = spre.common_reference(rec, reference="global", operator="median")
    return rec, notch_freqs


si.set_global_job_kwargs(chunk_duration="60s", n_jobs=1)

zarr_path = output_dir / "lfp.zarr"
meta_path = output_dir / "lfp_metadata.json"

if zarr_path.exists() and overwrite_cache:
    shutil.rmtree(zarr_path)

if zarr_path.exists():
    print(f"Reusing cache: {zarr_path}")
    lfp = se.read_zarr(zarr_path)
    cache_meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
else:
    print("Loading raw recording...")
    raw = load_maxwell_recording(data_path, well_id, rec_name=rec_name)
    print(f"  channels   : {raw.get_num_channels()}")
    print(f"  duration   : {raw.get_duration():.1f} s")
    print(f"  sample rate: {raw.get_sampling_frequency():.0f} Hz")

    print("Building LFP preprocessing chain...")
    lfp_lazy, applied_notches = build_lfp_chain(
        raw,
        target_fs_hz=lfp_target_fs_hz,
        electricity_hz=notch_electricity_hz,
        light_hz=notch_light_hz,
        q=notch_q,
        low_hz=lfp_low_hz,
        high_hz=lfp_high_hz,
    )

    print(f"Writing Zarr cache to {zarr_path}...")
    lfp_lazy.save(format="zarr", folder=zarr_path, progress_bar=True)
    lfp = se.read_zarr(zarr_path)

    cache_meta = {
        "data_path": str(data_path),
        "well_id": well_id,
        "rec_name": rec_name,
        "bandpass_hz": [lfp_low_hz, lfp_high_hz],
        "notch_electricity_hz": notch_electricity_hz,
        "notch_light_hz": notch_light_hz,
        "notch_q": notch_q,
        "notch_freqs_applied_hz": applied_notches,
        "lfp_target_fs_hz": lfp_target_fs_hz,
        "sampling_frequency_hz": float(lfp.get_sampling_frequency()),
        "num_channels": int(lfp.get_num_channels()),
        "duration_sec": float(lfp.get_duration()),
    }
    meta_path.write_text(json.dumps(cache_meta, indent=2))
    print("Done.")

fs = float(lfp.get_sampling_frequency())
duration_sec = float(lfp.get_duration())
print(f"\nLFP cache: {lfp.get_num_channels()} channels, {duration_sec:.1f} s @ {fs:.0f} Hz")


## 3. Interactive electrode selector

Click an electrode to toggle it. Drag a rectangle to add many. Use the text
box to paste explicit electrode IDs. The selected set is shared with the plot
cell below.

In [ ]:
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.widgets import RectangleSelector

from acute_slice_mea.recording import load_maxwell_recording
from acute_slice_mea.electrodes import build_electrode_table

# Probe geometry / electrode table comes from the raw recording's probe;
# the LFP cache only has channels, not the chip layout.
raw_for_probe = load_maxwell_recording(data_path, well_id, rec_name=rec_name)
electrodes = build_electrode_table(raw_for_probe.get_probe(), raw_for_probe)
recorded = electrodes[electrodes["recorded"].astype(bool)].copy()

contact_xy = electrodes[["x_um", "y_um"]].to_numpy()
recorded_eids = recorded["electrode_id"].astype(int).to_numpy()
recorded_xy = contact_xy[recorded_eids]

# Channel IDs come from the cached LFP — preserve whatever dtype Zarr
# round-tripped (Maxwell yields numpy.str_, but plot_traces needs the exact
# dtype back, so we map electrode_id -> the native channel object).
lfp_native_channels = list(lfp.get_channel_ids())
lfp_ch_by_str = {str(c): c for c in lfp_native_channels}

eid_to_channel = {}
for eid, ch in zip(recorded["electrode_id"].astype(int), recorded["channel_id"]):
    native = lfp_ch_by_str.get(str(ch))
    if native is not None:
        eid_to_channel[int(eid)] = native

recorded_eids = np.array(
    [eid for eid in recorded_eids if int(eid) in eid_to_channel],
    dtype=int,
)
recorded_xy = contact_xy[recorded_eids]

selected_eids: list[int] = []


def _selection_table(eids):
    rows = []
    for eid in eids:
        eid = int(eid)
        x_um, y_um = contact_xy[eid]
        rows.append({
            "electrode_id": eid,
            "channel_id": str(eid_to_channel[eid]),
            "x_um": float(x_um),
            "y_um": float(y_um),
        })
    return pd.DataFrame(rows, columns=["electrode_id", "channel_id", "x_um", "y_um"])


def _refresh():
    if selected_eids:
        selected_scatter.set_offsets(contact_xy[selected_eids])
    else:
        selected_scatter.set_offsets(np.empty((0, 2)))
    with status_out:
        status_out.clear_output()
        print(f"Selected {len(selected_eids)} electrode(s).")
        if selected_eids:
            display(_selection_table(selected_eids))
    fig.canvas.draw_idle()


def _set(eids, append=False):
    global selected_eids
    valid = [int(e) for e in eids if int(e) in eid_to_channel]
    if append:
        valid = selected_eids + valid
    selected_eids = sorted(dict.fromkeys(valid))
    _refresh()


def _toggle(eid):
    current = set(selected_eids)
    eid = int(eid)
    current.symmetric_difference_update({eid})
    _set(sorted(current))


def _on_pick(event):
    if event.artist is not recorded_scatter or len(event.ind) == 0:
        return
    mouse_xy = np.array([event.mouseevent.xdata, event.mouseevent.ydata], dtype=float)
    cand_xy = recorded_xy[event.ind]
    nearest = int(np.argmin(np.sum((cand_xy - mouse_xy) ** 2, axis=1)))
    _toggle(recorded_eids[event.ind[nearest]])


def _on_rect(eclick, erelease):
    if eclick.xdata is None or erelease.xdata is None:
        return
    x0, x1 = sorted([eclick.xdata, erelease.xdata])
    y0, y1 = sorted([eclick.ydata, erelease.ydata])
    mask = (
        (recorded_xy[:, 0] >= x0) & (recorded_xy[:, 0] <= x1) &
        (recorded_xy[:, 1] >= y0) & (recorded_xy[:, 1] <= y1)
    )
    _set(recorded_eids[mask], append=True)


fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(contact_xy[:, 0], contact_xy[:, 1], s=8, c="lightgray", marker="s",
           label="All contacts")
recorded_scatter = ax.scatter(
    recorded_xy[:, 0], recorded_xy[:, 1],
    s=40, c="tomato", marker="o",
    label="Recorded", picker=True, pickradius=5,
)
selected_scatter = ax.scatter(
    [], [], s=90, facecolors="none", edgecolors="royalblue",
    linewidths=2.0, label="Selected",
)
ax.set_title("Click to toggle · drag rectangle to add")
ax.set_xlabel("X (μm)"); ax.set_ylabel("Y (μm)")
ax.set_aspect("equal"); ax.grid(alpha=0.2); ax.legend(loc="best")

rect = RectangleSelector(
    ax, _on_rect, useblit=True, button=[1], interactive=True,
    props=dict(facecolor="royalblue", edgecolor="royalblue", alpha=0.15, fill=True),
)
fig.canvas.mpl_connect("pick_event", _on_pick)

manual_text = widgets.Text(value="", placeholder="e.g. 500, 501, 520",
                           description="Electrodes:",
                           layout=widgets.Layout(width="360px"))
set_btn = widgets.Button(description="Set", button_style="primary")
add_btn = widgets.Button(description="Append")
clr_btn = widgets.Button(description="Clear", button_style="warning")
status_out = widgets.Output()


def _parse(text):
    toks = text.replace("\n", ",").replace(";", ",").replace(" ", ",").split(",")
    return [t.strip() for t in toks if t.strip()]


set_btn.on_click(lambda _: _set(_parse(manual_text.value)))
add_btn.on_click(lambda _: _set(_parse(manual_text.value), append=True))
clr_btn.on_click(lambda _: _set([]))

display(widgets.HBox([manual_text, set_btn, add_btn, clr_btn]))
display(status_out)
_refresh()

## 4. Plot LFP traces

Uses `spikeinterface.widgets.plot_traces` on the cached LFP recording for the
currently selected electrodes.

In [ ]:
from spikeinterface.widgets import plot_traces

start_slider = widgets.FloatSlider(
    value=0, min=0, max=max(0.1, duration_sec - 1.0), step=1.0,
    description="Start (s):", layout=widgets.Layout(width="420px"),
)
dur_slider = widgets.FloatSlider(
    value=min(10.0, duration_sec), min=1.0, max=min(60.0, duration_sec), step=1.0,
    description="Duration:", layout=widgets.Layout(width="420px"),
)
mode_dd = widgets.Dropdown(
    options=["line", "map"], value="line", description="Mode:",
    layout=widgets.Layout(width="220px"),
)
go_btn = widgets.Button(description="Plot traces", button_style="success")
trace_out = widgets.Output()


def _do_plot(_=None):
    with trace_out:
        trace_out.clear_output(wait=True)
        if not selected_eids:
            print("No electrodes selected — use the chip plot above first.")
            return
        ch_ids = [eid_to_channel[int(e)] for e in selected_eids]
        t0 = float(start_slider.value)
        t1 = min(duration_sec, t0 + float(dur_slider.value))
        print(f"Plotting {len(ch_ids)} channel(s), t={t0:.1f}-{t1:.1f}s")
        w = plot_traces(
            lfp,
            channel_ids=ch_ids,
            time_range=(t0, t1),
            mode=mode_dd.value,
        )
        # plot_traces builds its own figure; display it explicitly so it lands
        # inside the Output widget under %matplotlib widget.
        fig_obj = getattr(w, "figure", None) or plt.gcf()
        display(fig_obj)


go_btn.on_click(_do_plot)

display(widgets.VBox([
    widgets.HBox([start_slider, dur_slider]),
    widgets.HBox([mode_dd, go_btn]),
]))
display(trace_out)